 QIC 861 — PostLab Assignment 1: Tomography
 ==============
 
#### By: Omer Sipra (20824310)


# Q1 - Draw a detailed experimental setup and describe the function of the important experimental components.

### Setup used throughout the lab
The optical chain used in all experiments was:

Laser → PBS1 → QWP1 → HWP1 → (process region: where an optic may be inserted) → HWP2 → QWP2 → PBS2 → Power meter

The polarization is treated as a single-qubit degree of freedom. The waveplates implement controlled polarization rotations, and the power meter provides a classical intensity readout that is proportional to projection probability up to an overall scale, provided we normalize appropriately within a measurement basis.

### What each component does

Laser  
Provides the optical beam used for polarization preparation and analysis. In practice, the absolute output can drift with time, and the output changes when optics are inserted or the system is rebuilt, so later analysis does not assume a fixed absolute power scale.

PBS1 - Polarizing Beam Splitter 1
Placed right after the laser. This defines the laboratory reference axis for horizontal polarization H. PBS1 is aligned first, and transmission is maximized through PBS1 so the input polarization is as close as possible to H. This fixes the meaning of H and V for the remainder of the lab.

Quarter-Waveplate and Half-Waveplate: QWP1 and HWP1 
These are the preparation waveplates. By setting their angles relative to calibrated zeros, they are used to prepare the standard polarization states H, V, D, A, R, and L. The order matters because waveplate operations do not commute, so the preparation stage is treated as a specific sequence QWP1 then HWP1 that take the Horizontal polarized pure state from PBS1 to any other state on the Bloch Sphere.

Process region (space for inserted element)  
A physical element can be inserted here to implement the polarization process or channel studied in later experiments, for example, a waveplate at a chosen angle, calcite beam displacer, partial polarizer, or no element for the identity process.

Half-Waveplate and Quarter-Waveplate: HWP2 and QWP2 (measurement / analyzer waveplates)  
These are the analyzer waveplates. They rotate the output polarization into the PBS2 basis so that PBS2 performs a projective measurement in the chosen basis. By selecting analyzer angles, PBS2 can effectively measure in the H/V, D/A, and R/L bases, corresponding to projections onto H, V, D, A, R, and L.

PBS2 (analyzer / second PBS)  
Acts as the final projective measurement device. After the analyzer waveplates set the measurement basis, PBS2 separates the two orthogonal outcomes of that basis. The power meter reads the corresponding intensity in the selected port.

Power meter  
Measures optical power in the selected output port. Because absolute power can drift and insertion losses can change, later analysis relies on normalization within each measurement basis rather than comparing raw powers across different analyzer settings.

### Reference max/min power levels and why we tracked them
Absolute power levels were not constant across the lab because inserting optics changes throughput and the laser output can drift. The lab notes therefore track reference high and low conditions as a stability and sanity check.

On the initial system we recorded a reference maximum transmission of about 5.15 mW, with a practical noise scale on the order of 1 µW. After inserting and setting QWPs to their calibrated zeros, transmitted power remained close to the maximum, about 5.05 mW, and after inserting both HWPs it remained similar, about 5.06 mW. This indicated that inserting and rotating these optics did not introduce a large, unexpected throughput change.

Later, the system was rebuilt because the week 1 laser was unreliable. On the rebuilt system we recorded a reference maximum transmission of about 2.518 mW and a minimum of about 0.54 µW. Although the absolute values differed strongly between systems, the purpose of these references was internal consistency: within a given setup, “near-max” and “near-min” conditions should remain stable and reproducible while data are taken. These references were used whenever waveplates were set to “zero” where we expect near-max transmission or to an orthogonal projection condition where we expect near-min transmission.

### Waveplate zero calibration and adopted zeros
Each waveplate’s “zero” was determined experimentally by scanning its dial angle and identifying the appropriate transmission extremum in the calibration configuration. Because peak-finding uses finite angle steps and finite meter precision, each zero has an associated uncertainty at the level of the mount resolution, repeatability, and the stability of the readout during scanning. In our notes, the rebuilt-system zeros were treated with an uncertainty scale of approximately ±1°.

After rebuilding and verifying consistency using the reference max/min checks, we adopted the following zeros for the full experiment chain and used them for all subsequent experiments:

$$
\mathrm{QWP1\ zero} = 0^\circ
$$

$$
\mathrm{HWP1\ zero} = -2^\circ
$$

$$
\mathrm{HWP2\ zero} = -2^\circ
$$

$$
\mathrm{QWP2\ zero} = -2^\circ
$$

All preparation and analyzer angles reported in later experiments are referenced consistently to these adopted zeros.

### Uncertainty in power measurements (least count model used in later analysis)
Our measurements are power readings, not photon counts. The power meter displayed values in mW with three decimal places, so the least count in that mode was 0.001 mW. We therefore assigned an uncertainty to each power measurement equal to half the least count, i.e. ±0.0005 mW, unless the readout was in the µW regime where the display similarly had three decimals in that unit, giving a least count of 0.001 µW and an uncertainty of ±0.0005 µW. This per-reading “half the last digit” rule was used consistently and provides the appropriate measurement variance needed later for maximum-likelihood reconstructions based on continuous power data, rather than Poisson counting statistics.


# Shared Utilities for Q2-Q6 Analysis

This section defines the several computational functions used for analysis. 



In [1]:
import numpy as np
import numpy.linalg as npl
from numpy.linalg import norm
from scipy.optimize import minimize

np.set_printoptions(precision=6, suppress=True)

np.set_printoptions(precision=6, suppress=True)

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1],[1, 0]], dtype=complex)
Y  = np.array([[0, -1j],[1j, 0]], dtype=complex)
Z  = np.array([[1, 0],[0, -1]], dtype=complex)
paulis = [I2, X, Y, Z]

ket_H = np.array([1,0], dtype=complex)
ket_V = np.array([0,1], dtype=complex)
ket_D = (ket_H + ket_V)/np.sqrt(2)
ket_A = (ket_H - ket_V)/np.sqrt(2)
ket_R = (ket_H + 1j*ket_V)/np.sqrt(2)
ket_L = (ket_H - 1j*ket_V)/np.sqrt(2)

def proj(ket):
    ket = ket.reshape(2,1)
    return ket @ ket.conj().T

projs_6 = {  # useful for Q2/Q3
    "H": proj(ket_H), "V": proj(ket_V),
    "D": proj(ket_D), "A": proj(ket_A),
    "R": proj(ket_R), "L": proj(ket_L),
}

def make_hermitian(A):
    return 0.5*(A + A.conj().T)

def purity(rho):
    rhoH = make_hermitian(rho)
    return float(np.real(np.trace(rhoH @ rhoH)))

def fidelity_pure(rho, psi):
    """F = <psi|rho|psi>"""
    rhoH = make_hermitian(rho)
    return float(np.real(np.conjugate(psi) @ (rhoH @ psi)))

def sqrtm_psd(A):
    AH = make_hermitian(A)
    w, v = npl.eigh(AH)
    w = np.clip(w, 0.0, None)
    return (v * np.sqrt(w)) @ v.conj().T

def fidelity_uhlmann(rho, sigma):
    rhoH   = make_hermitian(rho)
    sigmaH = make_hermitian(sigma)
    s = sqrtm_psd(sigmaH)
    M = s @ rhoH @ s
    return float(np.real(np.trace(sqrtm_psd(M)))**2)

def bloch_from_probs(pH, pD, pR):
    return np.array([2*pD - 1, 2*pR - 1, 2*pH - 1], dtype=float)

def rho_from_bloch(r):
    x,y,z = r
    return 0.5*(I2 + x*X + y*Y + z*Z)

def project_to_physical(rho, eps=0.0):
    rhoH = make_hermitian(rho)
    w, V = npl.eigh(rhoH)
    w = np.clip(w, eps, None)
    w = w / np.sum(w)
    return V @ np.diag(w) @ V.conj().T

def summarize_state(rho, psi_target=None, label=""):
    rhoH = make_hermitian(rho)
    eigs = np.real_if_close(npl.eigvalsh(rhoH))
    out = {
        "label": label,
        "rho": rhoH,
        "eigs": eigs,
        "purity": purity(rhoH),
    }
    if psi_target is not None:
        out["fidelity"] = fidelity_pure(rhoH, psi_target)
    return out

def _fmt(x, nd=6):
    if isinstance(x, (float, int, np.floating, np.integer)):
        return f"{float(x):.{nd}f}"
    return x

def print_header(title):
    bar = "=" * len(title)
    print(f"\n{title}\n{bar}")

def print_kv(d, nd=6, indent=2):
    pad = max(len(str(k)) for k in d.keys())
    for k, v in d.items():
        if isinstance(v, (float, int, np.floating, np.integer)):
            v_str = _fmt(v, nd=nd)
        elif isinstance(v, np.ndarray):
            v_str = np.array2string(v, precision=nd, suppress_small=True)
        else:
            v_str = str(v)
        print(" " * indent + f"{str(k):<{pad}} : {v_str}")

def print_matrix(M, name="Matrix", nd=6):
    print(f"{name}:")
    print(np.array2string(M, precision=nd, suppress_small=True))

def state_report(label, rho, probs=None, bloch=None, eigs=None, purity_val=None, fid=None, nd=6):
    """
    Standard state-tomography report..
    """
    print_header(label)
    if probs is not None:
        print("Probabilities")
        print_kv(probs, nd=nd)
    if bloch is not None:
        print("\nBloch vector")
        print_kv({"r": bloch, "|r|": float(npl.norm(bloch))}, nd=nd)
    print()
    print_matrix(rho, name="rho", nd=nd)
    if eigs is not None:
        print("\nEigenvalues")
        print_kv({"eigs": np.array(eigs)}, nd=nd)
    if purity_val is not None:
        print("\nPurity")
        print_kv({"Tr(rho^2)": purity_val}, nd=nd)
    if fid is not None:
        print("\nFidelity")
        print_kv({"F": fid}, nd=nd)

def compare_meas_pred(title, measured, predicted, nd=6):
    """
    Clean comparison printout for measured vs predicted scalars.
    measured/predicted: dicts with identical keys.
    """
    print_header(title)
    keys = list(measured.keys())
    pad = max(len(str(k)) for k in keys)
    for k in keys:
        m = measured[k]
        p = predicted[k]
        err = (m - p)
        rel = (err / m) if (isinstance(m,(int,float,np.floating)) and m!=0) else np.nan
        print(f"  {str(k):<{pad}}  meas={_fmt(m,nd)}   pred={_fmt(p,nd)}   diff={_fmt(err,nd)}   rel={_fmt(rel,nd)}")


SIGMA_PWR = 0.0005  

def summarize(x):
    x = np.array(x, dtype=float)
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x, ddof=1)),
        "p16": float(np.percentile(x, 16)),
        "p50": float(np.percentile(x, 50)),
        "p84": float(np.percentile(x, 84)),
    }

def pm(s, nd=6):
    p50 = s["p50"]
    lo = p50 - s["p16"]
    hi = s["p84"] - p50
    return f"{p50:.{nd}f} (-{lo:.{nd}f}, +{hi:.{nd}f})"

def noisy_scalar(x, rng, sigma=SIGMA_PWR):
    return max(0.0, float(x + rng.normal(0.0, sigma)))

def noisy_like(obj, rng, sigma=SIGMA_PWR):
    if isinstance(obj, (float, int, np.floating, np.integer)):
        return noisy_scalar(obj, rng, sigma=sigma)
    if isinstance(obj, dict):
        return {k: noisy_like(v, rng, sigma=sigma) for k, v in obj.items()}
    raise TypeError(f"Unsupported type for noisy_like: {type(obj)}")

def bootstrap_report(raw, metrics_fn, nboot=2000, seed=0, sigma=SIGMA_PWR):
    m0 = metrics_fn(raw)

    rng = np.random.default_rng(seed)
    samples = {k: [] for k in m0.keys()}

    for _ in range(nboot):
        rawp = noisy_like(raw, rng, sigma=sigma)
        mk = metrics_fn(rawp)
        for k in samples.keys():
            samples[k].append(float(mk[k]))

    ms = {k: summarize(v) for k, v in samples.items()}
    return m0, ms


## Q2 - Quantum State Tomography of Single Photon Pure States

### Method summary
For each reconstructed qubit state $\rho$ we report:
- **Fidelity** with target pure state $|\psi\rangle$:  $F = \langle \psi | \rho | \psi \rangle$
- **Purity**:  $\gamma = \mathrm{Tr}(\rho^2)$

### How probabilities are obtained from power
- When both outcomes of a basis are measured (e.g., H and V), we use **pair-normalization**
  $$p(H)=\ \frac{P_H}{P_H+P_V}$$
- If only one projector power is available, we map power into a probability using reference min/max:
  $$p = \ \frac{P - P_{\\min}}{P_{\\max}-P_{\\min}}$$
  (and then infer the complement by $1-p$).

Finally, we construct the Bloch vector $r=(x,y,z)$ from $(p_H,p_D,p_R)$, form $\\rho$, and project to a physical density matrix if the linear estimate falls slightly outside the Bloch ball due to experimental noise.


### (a) Present your results from performing maximum likelihood state tomography on the prepared |H⟩ state. What is the fidelity of the reconstructed state? What is the purity?

Data recorded: analyzer settings H, V, D, R.

- **H/V:** compute $p_H$ by pair-normalization.
- **D and R:** convert single-port power into a probability using the min/max reference mapping (since the complementary port was not explicitly measured in that part of the procedure).

Then:
1. build the Bloch vector from $(p_H,p_D,p_R)$
2. compute the linear inversion density matrix
3. project it onto the physical set (PSD, trace-1)
4. report fidelity with $|H\rangle$ and purity.

In [2]:
P_H = 2.415
P_V = 4.69e-3   # 4.69 µW -> mW
P_D = 1.210
P_R = 1.136

P_max = 2.518
P_min = 0.54e-3  # 0.54 µW -> mW

Q2a_raw = {
    "P_H": float(P_H),
    "P_V": float(P_V),
    "P_D": float(P_D),
    "P_R": float(P_R),
    "P_max": float(P_max),
    "P_min": float(P_min),
}


pH = P_H/(P_H + P_V)
pD = np.clip((P_D - P_min)/(P_max - P_min), 0.0, 1.0)
pR = np.clip((P_R - P_min)/(P_max - P_min), 0.0, 1.0)

r = bloch_from_probs(pH, pD, pR)
rho_lin = rho_from_bloch(r)
rho = project_to_physical(rho_lin)

summary_H = summarize_state(rho, psi_target=ket_H, label="Q2(a) rho_H")

state_report(
    label="Q2(a) Prepared |H> — reconstructed state",
    rho=summary_H["rho"],
    probs={"pH": pH, "pD": pD, "pR": pR},
    bloch=r,
    eigs=summary_H["eigs"],
    purity_val=summary_H["purity"],
    fid=summary_H["fidelity"],
    nd=6
)

Q2a_raw = {
    "P_H": float(P_H),
    "P_V": float(P_V),
    "P_D": float(P_D),
    "P_R": float(P_R),
    "P_max": float(P_max),
    "P_min": float(P_min),
}



Q2(a) Prepared |H> — reconstructed state
Probabilities
  pH : 0.998062
  pD : 0.480429
  pR : 0.451034

Bloch vector
  r   : [-0.039143 -0.097932  0.996123]
  |r| : 1.001691

rho:
[[ 0.997221+0.j       -0.019538+0.048883j]
 [-0.019538-0.048883j  0.002779+0.j      ]]

Eigenvalues
  eigs : [0. 1.]

Purity
  Tr(rho^2) : 1.000000

Fidelity
  F : 0.997221


In [3]:
def metrics_Q2a(raw):
    PH, PV = raw["P_H"], raw["P_V"]
    PD, PR = raw["P_D"], raw["P_R"]
    Pmx, Pmn = raw["P_max"], raw["P_min"]

    pH = PH/(PH+PV) if (PH+PV) > 1e-15 else 0.5
    denom_mm = Pmx - Pmn
    if abs(denom_mm) < 1e-15:
        pD = 0.5
        pR = 0.5
    else:
        pD = float(np.clip((PD - Pmn)/denom_mm, 0.0, 1.0))
        pR = float(np.clip((PR - Pmn)/denom_mm, 0.0, 1.0))

    r = bloch_from_probs(pH, pD, pR)
    rho = project_to_physical(rho_from_bloch(r))
    rho = make_hermitian(rho)

    return {
        "F": fidelity_pure(rho, ket_H),
        "purity": purity(rho),
        "|r|": float(norm(r)),
        "pH": float(pH),
        "pD": float(pD),
        "pR": float(pR),
    }

m0, ms = bootstrap_report(Q2a_raw, metrics_Q2a, nboot=20000, seed=1)

print_header("Uncertainty — Q2(a) Prepared |H>")
print("Noise-free:")
print_kv(m0, nd=6)

print("\nBootstrap (median ± 16/84):")
for k in ["F","purity","|r|","pH","pD","pR"]:
    print(f"  {k:>6} : {pm(ms[k], nd=6)}")



Uncertainty — Q2(a) Prepared |H>
Noise-free:
  F      : 0.997221
  purity : 1.000000
  |r|    : 1.001691
  pH     : 0.998062
  pD     : 0.480429
  pR     : 0.451034

Bootstrap (median ± 16/84):
       F : 0.997220 (-0.000028, +0.000027)
  purity : 1.000000 (-0.000000, +0.000000)
     |r| : 1.001693 (-0.000406, +0.000412)
      pH : 0.998062 (-0.000203, +0.000205)
      pD : 0.480421 (-0.000235, +0.000234)
      pR : 0.451032 (-0.000244, +0.000233)


#### Analysis

From the tomography probabilities
- $p_H = 0.998062$ (HV basis),
- $p_D = 0.480429$ (DA basis),
- $p_R = 0.451034$ (RL basis),

the inferred Bloch vector is
$$r = (x,y,z) = (2p_D-1,\;2p_R-1,\;2p_H-1) = (-0.039143,\;-0.097932,\;0.996123).$$
This is very close to the $+z$ axis, so the reconstructed state is strongly aligned with $|H⟩$ as expected.

The reconstructed density matrix is
$$
\rho =
\begin{pmatrix}
0.997221 & -0.019538 + 0.048883\,i \\
-0.019538 - 0.048883\,i & 0.002779
\end{pmatrix},
$$
which shows:
- the population is overwhelmingly in $|H⟩$ (top-left element $\approx 0.997$),
- a small residual off-diagonal coherence, consistent with a nearly pure state slightly tilted away from the exact $z$ axis.

**Fidelity and purity.**  
The state fidelity with the target $|H⟩$ is
$$F=\langle H|\rho|H\rangle = 0.997221,$$
meaning the prepared state is within about $0.3\%$ infidelity of the ideal $|H⟩$.  
The purity is
$$\mathrm{Tr}(\rho^2)=1.000000,$$
consistent with an essentially pure state (the eigenvalues are reported as $(0,1)$ within numerical precision).

**Comment on $|r|>1$.**  
The Bloch norm is reported as $|r|=1.001691$, slightly above 1. This is unphysical and indicates a small systematic/statistical overshoot from finite measurement noise and calibration (a common artifact when reconstructing $r$ directly from probabilities). In the uncertainty analysis we explicitly project $\rho$ to the closest physical state, which is why purity stays exactly at 1 while $|r|$ fluctuates slightly.

**Uncertainty (bootstrap over power noise).**  
Using $\sigma=0.0005$ mW power-meter noise and resampling the raw powers, the key quantities are extremely stable:
- $F = 0.997220^{+0.000027}_{-0.000028}$,
- $p_H = 0.998062^{+0.000205}_{-0.000203}$,
- $p_D = 0.480421^{+0.000234}_{-0.000235}$,
- $p_R = 0.451032^{+0.000233}_{-0.000244}$.

These uncertainties are at the $10^{-4}$ level, far smaller than the deviation from the ideal probabilities themselves, so the conclusion is robust: the preparation is a high-fidelity, near-pure $|H⟩$ state.


### (b) Show the results from performing maximum likelihood state tomography on the prepared |R⟩ state. Whit is the fidelity? What is the Purity? Show a comparison between explicitly measuring |A⟩ and |L⟩ in the lab, and the expected values obtained from your reconstructed density matrix.

From the Exp 2 prepared-R table, we recorded powers for all six projectors H, V, D, A, R, L.

We compute $p_H,p_D,p_R$ by pair-normalizing within each basis pair (H,V), (D,A), and (R,L), then reconstruct $\rho_R$.

Target state:
$$|R\rangle=\frac{1}{\sqrt{2}}(|H\rangle+i|V\rangle).$$

We report purity $\mathrm{Tr}(\rho_R^2)$ and fidelity $F=\langle R|\rho_R|R\rangle$.

In [4]:
P_H = 1.054
P_V = 1.229
P_D = 1.197
P_A = 1.124
P_R = 2.403
P_L = 8.27e-3  # 8.27 µW -> mW

Q2b_raw = {
    "P_H": float(P_H), "P_V": float(P_V),
    "P_D": float(P_D), "P_A": float(P_A),
    "P_R": float(P_R), "P_L": float(P_L),
}


pH = P_H/(P_H + P_V)
pD = P_D/(P_D + P_A)
pR = P_R/(P_R + P_L)

r = bloch_from_probs(pH, pD, pR)
rho_R = rho_from_bloch(r)

summary_R = summarize_state(rho_R, psi_target=ket_R, label="Q2(b) rho_R")

state_report(
    label="Q2(b) Prepared |R> — reconstructed state",
    rho=summary_R["rho"],
    probs={"pH": pH, "pD": pD, "pR": pR},
    bloch=r,
    eigs=summary_R["eigs"],
    purity_val=summary_R["purity"],
    fid=summary_R["fidelity"],
    nd=6
)



Q2(b) Prepared |R> — reconstructed state
Probabilities
  pH : 0.461673
  pD : 0.515726
  pR : 0.996570

Bloch vector
  r   : [ 0.031452  0.993141 -0.076654]
  |r| : 0.996591

rho:
[[0.461673+0.j      0.015726-0.49657j]
 [0.015726+0.49657j 0.538327+0.j     ]]

Eigenvalues
  eigs : [0.001705 0.998295]

Purity
  Tr(rho^2) : 0.996597

Fidelity
  F : 0.996570


In [5]:
def metrics_Q2b(raw):
    PH, PV = raw["P_H"], raw["P_V"]
    PD, PA = raw["P_D"], raw["P_A"]
    PR, PL = raw["P_R"], raw["P_L"]

    pH = PH/(PH+PV) if (PH+PV) > 1e-15 else 0.5
    pD = PD/(PD+PA) if (PD+PA) > 1e-15 else 0.5
    pR = PR/(PR+PL) if (PR+PL) > 1e-15 else 0.5

    r = bloch_from_probs(pH, pD, pR)
    rho = project_to_physical(rho_from_bloch(r))
    rho = make_hermitian(rho)

    return {
        "F": fidelity_pure(rho, ket_R),
        "purity": purity(rho),
        "|r|": float(norm(r)),
        "pH": float(pH),
        "pD": float(pD),
        "pR": float(pR),
    }

m0, ms = bootstrap_report(Q2b_raw, metrics_Q2b, nboot=20000, seed=2)

print_header("Uncertainty — Q2(b) Prepared |R>")
print("Noise-free (same code path):")
print_kv(m0, nd=6)

print("\nBootstrap (median ± 16/84):")
for k in ["F","purity","|r|","pH","pD","pR"]:
    print(f"  {k:>6} : {pm(ms[k], nd=6)}")



Uncertainty — Q2(b) Prepared |R>
Noise-free (same code path):
  F      : 0.996570
  purity : 0.996597
  |r|    : 0.996591
  pH     : 0.461673
  pD     : 0.515726
  pR     : 0.996570

Bootstrap (median ± 16/84):
       F : 0.996570 (-0.000202, +0.000206)
  purity : 0.996598 (-0.000401, +0.000409)
     |r| : 0.996592 (-0.000402, +0.000410)
      pH : 0.461675 (-0.000159, +0.000153)
      pD : 0.515726 (-0.000150, +0.000149)
      pR : 0.996570 (-0.000202, +0.000206)


#### Analysis

From the tomography probabilities
- $p_H = 0.461673$ (HV basis),
- $p_D = 0.515726$ (DA basis),
- $p_R = 0.996570$ (RL basis),

the inferred Bloch vector is
$$r = (2p_D-1,\;2p_R-1,\;2p_H-1) = (0.031452,\;0.993141,\;-0.076654),$$
with magnitude $|r|=0.996591$. The dominant component is the $y$-component ($\approx 0.993$), which is exactly the axis associated with the $R/L$ basis. This indicates the prepared state is very close to the ideal right-circular polarization state $|R⟩$.

The reconstructed density matrix is
$$
\rho =
\begin{pmatrix}
0.461673 & 0.015726 - 0.496570\,i \\
0.015726 + 0.496570\,i & 0.538327
\end{pmatrix}.
$$
The large imaginary off-diagonal term ($\approx \mp 0.497\,i$) is the signature of circular polarization (relative phase of $\pm \pi/2$ between $|H⟩$ and $|V⟩$), consistent with a near-$|R⟩$ state. The diagonal imbalance ($\rho_{VV}>\rho_{HH}$) corresponds to the small negative $z$ component ($-0.0767$), i.e., a slight tilt away from the equator.

**Fidelity and mixedness.**  
The fidelity with the target $|R⟩$ is
$$F=\langle R|\rho|R\rangle = 0.996570,$$
so the state is within about $0.34\%$ infidelity of ideal.  
The eigenvalues are $(0.001705,\;0.998295)$, indicating a small admixture of the orthogonal state (about $0.17\%$ weight). This agrees with the purity:
$$\mathrm{Tr}(\rho^2)=0.996597,$$
which is slightly below 1, meaning the state is very nearly pure but not perfectly.

**Interpretation of the measured probabilities.**  
Even though $p_H$ is not close to 0.5 (it is $0.4617$), this does not contradict an $|R⟩$ preparation: the defining feature of $|R⟩$ is that it is an eigenstate of the $R/L$ measurement, and indeed $p_R=0.996570$ is almost unity. The small deviations in $p_H$ and $p_D$ quantify the small tilt away from the ideal $+y$ axis.

**Uncertainty (bootstrap over power noise).**  
Using $\sigma=0.0005$ mW power-meter noise, the reconstruction is highly stable:
- $F = 0.996570^{+0.000206}_{-0.000202}$,
- $\mathrm{Tr}(\rho^2)=0.996598^{+0.000409}_{-0.000401}$,
- $|r|=0.996592^{+0.000410}_{-0.000402}$,
- $p_R = 0.996570^{+0.000206}_{-0.000202}$.

These uncertainties are at the $10^{-4}$ level, so the conclusion is robust: the prepared state is a high-fidelity $|R⟩$ state with a small, quantifiable amount of depolarization / admixture.


## Q3 - Quantum State Tomography of Mixed States

In Experiment 3 we prepare $|R\rangle$ and insert the calcite beam displacer. The displacer couples polarization to spatial mode (path). If we do not resolve the path degree of freedom and instead detect total power from both beams, the polarization subsystem is described by a reduced density matrix that can become mixed. We reconstruct the output polarization density matrix and compare it to the expected mixed state.


#### (a) Show your reconstructed density matrix corresponding to preparing |R⟩ and passing thephotons through the calcite beam displacer. What quantum state does this prepare? What is the fidelity of the preparation and what is the purity?

We prepare
$$|R\rangle=\frac{1}{\sqrt{2}}\left(|H\rangle+i|V\rangle\right).$$

An ideal beam displacer separates the $|H\rangle$ and $|V\rangle$ polarization components into different transverse paths. Denote the two spatial modes by $|0\rangle_{\text{path}}$ and $|1\rangle_{\text{path}}$. After the beam displacer, the joint polarization–path state is
$$|\Psi\rangle=\frac{1}{\sqrt{2}}\left(|H\rangle\otimes|0\rangle_{\text{path}} + i\,|V\rangle\otimes|1\rangle_{\text{path}}\right).$$

This is an entangled state between polarization and path. If our detector collects both beams without resolving which path occurred, then the polarization state is described by the reduced density matrix
$$\rho_{\text{pol}}=\mathrm{Tr}_{\text{path}}\left(|\Psi\rangle\langle\Psi|\right).$$

Computing the partial trace removes the cross terms because the paths are orthogonal:
$$\mathrm{Tr}_{\text{path}}\left(|0\rangle\langle1|\right)=\mathrm{Tr}_{\text{path}}\left(|1\rangle\langle0|\right)=0,$$
so the polarization coherence between $|H\rangle$ and $|V\rangle$ is lost. The result is
$$\rho_{\text{pol}}=\frac{1}{2}\left(|H\rangle\langle H| + |V\rangle\langle V|\right)=\frac{I}{2}.$$

Therefore, even though the state $|\Psi\rangle$ is pure, the polarization subsystem becomes maximally mixed when the path information is not observed. Functionally, this looks like a dephasing mechanism in the $H/V$ basis: populations remain, but relative phase information is destroyed.


### (b) Discuss your observations with and without the beam displacer inserted by examining the differences between coherent superpositions of pure states, and probabilistic mixtures of pure states.

We perform the same single-qubit reconstruction as in Q2 using the full six-projector dataset (H,V,D,A,R,L) taken with the beam displacer inserted.

To compare to the expected state $I/2$, we report:
- Pair-normalized probabilities in each basis (should be close to 0.5)
- Purity $\gamma=\mathrm{Tr}(\rho^2)$ (ideal for $I/2$ is $\gamma=0.5$)
- Fidelity with the mixed target state $I/2$ using the mixed-state (Uhlmann) fidelity
$$F(\rho,\sigma)=\left(\mathrm{Tr}\sqrt{\sqrt{\sigma}\rho\sqrt{\sigma}}\right)^2.$$


In [6]:
P_H = 1.086
P_V = 1.270
P_D = 1.180
P_A = 1.179
P_R = 1.176
P_L = 1.184

Q3b_raw = {
    "P_H": float(P_H), "P_V": float(P_V),
    "P_D": float(P_D), "P_A": float(P_A),
    "P_R": float(P_R), "P_L": float(P_L),
}


pH = P_H/(P_H + P_V)
pD = P_D/(P_D + P_A)
pR = P_R/(P_R + P_L)

r3 = bloch_from_probs(pH, pD, pR)
rho3 = rho_from_bloch(r3)

rho_mix = 0.5 * I2

pur3 = purity(make_hermitian(rho3))
F3 = fidelity_uhlmann(rho3, rho_mix)  # mixed-state fidelity

state_report(
    label="Q3(b) Prepared |R> + beam displacer — reconstructed state",
    rho=make_hermitian(rho3),
    probs={
        "pH (H|HV)": pH, "pV (V|HV)": 1-pH,
        "pD (D|DA)": pD, "pA (A|DA)": 1-pD,
        "pR (R|RL)": pR, "pL (L|RL)": 1-pR,
    },
    bloch=r3,
    eigs=np.real_if_close(npl.eigvalsh(make_hermitian(rho3))),
    purity_val=pur3,
    fid=None,
    nd=6
)

print_header("Comparison to ideal mixed state I/2")
print_kv({
    "purity Tr(rho^2) (ideal 0.5)": pur3,
    "F(rho, I/2) (ideal 1)": F3
}, nd=6)



Q3(b) Prepared |R> + beam displacer — reconstructed state
Probabilities
  pH (H|HV) : 0.460951
  pV (V|HV) : 0.539049
  pD (D|DA) : 0.500212
  pA (A|DA) : 0.499788
  pR (R|RL) : 0.498305
  pL (L|RL) : 0.501695

Bloch vector
  r   : [ 0.000424 -0.00339  -0.078098]
  |r| : 0.078173

rho:
[[0.460951+0.j       0.000212+0.001695j]
 [0.000212-0.001695j 0.539049+0.j      ]]

Eigenvalues
  eigs : [0.460913 0.539087]

Purity
  Tr(rho^2) : 0.503056

Comparison to ideal mixed state I/2
  purity Tr(rho^2) (ideal 0.5) : 0.503056
  F(rho, I/2) (ideal 1)        : 0.998470


In [7]:
def metrics_Q3b(raw):
    PH, PV = raw["P_H"], raw["P_V"]
    PD, PA = raw["P_D"], raw["P_A"]
    PR, PL = raw["P_R"], raw["P_L"]

    pH = PH/(PH+PV) if (PH+PV) > 1e-15 else 0.5
    pD = PD/(PD+PA) if (PD+PA) > 1e-15 else 0.5
    pR = PR/(PR+PL) if (PR+PL) > 1e-15 else 0.5

    r = bloch_from_probs(pH, pD, pR)
    rho = project_to_physical(rho_from_bloch(r))
    rho = make_hermitian(rho)

    return {
        "F_I2": fidelity_uhlmann(rho, 0.5*I2),
        "purity": purity(rho),
        "|r|": float(norm(r)),
    }

m0, ms = bootstrap_report(Q3b_raw, metrics_Q3b, nboot=20000, seed=3)

print_header("Uncertainty — Q3(b) Mixed state (systematic)")
print("Noise-free (same code path):")
print_kv(m0, nd=6)

print("\nBootstrap (median ± 16/84):")
for k in ["F_I2","purity","|r|"]:
    print(f"  {k:>6} : {pm(ms[k], nd=6)}")



Uncertainty — Q3(b) Mixed state (systematic)
Noise-free (same code path):
  F_I2   : 0.998470
  purity : 0.503056
  |r|    : 0.078173

Bootstrap (median ± 16/84):
    F_I2 : 0.998470 (-0.000012, +0.000012)
  purity : 0.503056 (-0.000023, +0.000023)
     |r| : 0.078175 (-0.000300, +0.000296)


####  Analysis 

In this part, the input polarization is prepared as $|R⟩$, then passed through the beam displacer. The beam displacer couples polarization to spatial path. If the path information is not erased (i.e., we trace over path), polarization coherence is expected to be lost, producing a state close to the maximally mixed state $I/2$.

#### Measured probabilities and Bloch vector
The measured probabilities in each basis are essentially balanced:
- HV: $p_H=0.460951$, $p_V=0.539049$
- DA: $p_D=0.500212$, $p_A=0.499788$
- RL: $p_R=0.498305$, $p_L=0.501695$

This yields the Bloch vector
$$r=(x,y,z)=(0.000424,\;-0.003390,\;-0.078098),$$
with magnitude
$$|r|=0.078173.$$
Since $|r|\ll 1$, the polarization state is very close to being maximally mixed. The small nonzero $z$ component indicates a slight residual bias toward $|V⟩$ over $|H⟩$ (consistent with $p_V>p_H$).

#### Reconstructed density matrix and spectrum
The reconstructed density matrix is
$$
\rho=
\begin{pmatrix}
0.460951 & 0.000212 + 0.001695\,i\\
0.000212 - 0.001695\,i & 0.539049
\end{pmatrix}.
$$
The off-diagonal terms are extremely small, meaning coherence between $|H⟩$ and $|V⟩$ has been almost completely suppressed. The eigenvalues are
$$(0.460913,\;0.539087),$$
which are close to the ideal mixed-state eigenvalues $(0.5,0.5)$, again indicating near-maximal mixing.

#### Purity and fidelity to $I/2$
The purity is
$$\mathrm{Tr}(\rho^2)=0.503056,$$
very close to the ideal maximally mixed value $0.5$. The slight excess over $0.5$ corresponds to the small residual Bloch magnitude $|r|\approx 0.078$ (i.e., the state is almost, but not perfectly, maximally mixed).

The Uhlmann fidelity with the ideal maximally mixed state is
$$F(\rho, I/2)=0.998470,$$
so the observed polarization state after the beam displacer is extremely close to $I/2$.

#### Physical interpretation
This result matches the expected role of the beam displacer: it converts polarization information into path information. When the path degree of freedom is not observed, the polarization subsystem undergoes effective decoherence and becomes nearly maximally mixed. The remaining small bias (mostly along $z$) is consistent with a slight imbalance in splitting/transmission rather than residual polarization coherence.

#### Uncertainty
Bootstrap resampling with $\sigma=0.0005$ mW power noise shows the conclusion is highly robust:
- $F(\rho,I/2)=0.998470^{+0.000012}_{-0.000012}$
- $\mathrm{Tr}(\rho^2)=0.503056^{+0.000023}_{-0.000023}$
- $|r|=0.078175^{+0.000296}_{-0.000300}$

The uncertainties are much smaller than the observed deviation from the ideal values, so the small residual bias away from perfect mixing is systematic, not statistical noise.


## Q4 - Single Photon Process Tomography of Trace-Preserving Unitary Channels

We reconstruct a single-qubit quantum process $\mathcal{E}$ acting on polarization using the process matrix $\chi$ in the Pauli basis $\{I,X,Y,Z\}$:
$$\mathcal{E}(\rho)=\sum_{m,n\in\{I,X,Y,Z\}} \chi_{mn}\,\sigma_m\,\rho\,\sigma_n^\dagger.$$

The experiment records optical powers $P$ proportional to projective measurement probabilities. For each prepared input state $\rho_\text{in}$ and measurement projector $\Pi_m$, the model is:
$$P_{m,\text{meas}}(\rho_\text{in}) \approx \alpha_{\rho_\text{in}}\;\mathrm{Tr}\!\left(\Pi_m\,\mathcal{E}(\rho_\text{in})\right),$$
where $\alpha_{\rho_\text{in}}$ is a per-input scale factor absorbing throughput differences between preparations.

We fit $\chi$ by maximum-likelihood / weighted least squares with:
- $\chi \succeq 0$ enforced by parametrizing $\chi = A A^\dagger / \mathrm{Tr}(A A^\dagger)$
- $\mathrm{Tr}(\chi)=1$ by construction
- trace preservation enforced with a penalty term on the TP residual.

We then interpret the fitted $\chi$ by:
- examining its eigenvalues,
- extracting a nearest unitary $U_\text{fit}$ from the dominant eigenvector of $\chi$,
- extracting axis–angle parameters $(\phi,\hat n)$ and comparing $\phi$ to the ideal HWP value ($180^\circ$).


We use only the four preparations and four measurement projectors: $H,V,D,R$.

Implementation note: we already defined the 6 projectors in `projs_6` above; here we just reuse the subset `{H,V,D,R}` for process tomography.


In [8]:
preps = ["H", "V", "D", "R"]
meas  = ["H", "V", "D", "R"]

projs_4 = {k: projs_6[k] for k in preps}  # re-use existing projectors

E = [I2, X, Y, Z]


#### Applying a $\chi$-process and compute predicted probabilities

For a given $\chi$, the predicted probability is
$$p(m|\rho_\text{in})=\mathrm{Tr}\!\left(\Pi_m\,\mathcal{E}(\rho_\text{in})\right).$$

We use this to build a weighted least-squares negative log-likelihood in powers, including per-preparation scale factors $\alpha_{\rho_\text{in}}$.


In [9]:
def apply_process_chi(chi, rho):
    out = np.zeros((2,2), dtype=complex)
    for m in range(4):
        for n in range(4):
            out += chi[m, n] * (E[m] @ rho @ E[n].conj().T)
    return out

def born_prob(chi, prep_label, meas_label):
    rho_in = projs_4[prep_label]
    Pi = projs_4[meas_label]
    rho_out = apply_process_chi(chi, rho_in)
    return float(np.real(np.trace(Pi @ rho_out)))


#### Trace-preservation penalty

Trace preservation is equivalent to:
$$\sum_{m,n}\chi_{mn}\,\sigma_n^\dagger\sigma_m = I.$$

We enforce this softly by adding a penalty proportional to the Frobenius norm squared of the TP residual.


In [10]:
def tp_residual(chi):
    S = np.zeros((2,2), dtype=complex)
    for m in range(4):
        for n in range(4):
            S += chi[m, n] * (E[n].conj().T @ E[m])
    return S - I2


#### PSD + trace-1 parameterization for $\chi$

We parametrize $\chi$ as
$$\chi = \frac{A A^\dagger}{\mathrm{Tr}(A A^\dagger)}$$
with an unconstrained complex $4\times 4$ matrix $A$. This guarantees $\chi \succeq 0$ and $\mathrm{Tr}(\chi)=1$ automatically.

The optimizer therefore searches over real parameters encoding the real and imaginary parts of $A$.


In [11]:
def unpack_params_to_chi(theta):
    A = np.zeros((4,4), dtype=complex)
    idx = 0
    for i in range(4):
        for j in range(4):
            re = theta[idx]; im = theta[idx+1]
            A[i, j] = re + 1j*im
            idx += 2
    chi = A @ A.conj().T
    tr = float(np.real(np.trace(chi)))
    if tr <= 1e-15:
        tr = 1e-15
    return chi / tr

SIGMA = SIGMA_PWR


In [12]:
def best_alphas_for_chi(chi, dataset):
    alphas = {}
    for prep in preps:
        Ps = np.array([dataset[prep][m] for m in meas], dtype=float)
        ps = np.array([born_prob(chi, prep, m) for m in meas], dtype=float)

        num = np.sum(Ps * ps)
        den = np.sum(ps * ps) + 1e-15
        alphas[prep] = num / den
    return alphas


#### Objective function and MLE fit

We minimize:
- squared residuals between measured powers and predicted powers,
- plus a TP penalty term.

The only role of the optimizer is to find the best $\chi$; everything else (physicality, normalization, per-prep scale) is handled analytically or by construction.


In [13]:
def loss_for_theta(theta, dataset, lam_tp=1e4):
    chi = unpack_params_to_chi(theta)
    alphas = best_alphas_for_chi(chi, dataset)

    L = 0.0
    for prep in preps:
        alpha = alphas[prep]
        for m in meas:
            P = dataset[prep][m]
            p = born_prob(chi, prep, m)
            pred = alpha * p
            L += ((P - pred)/SIGMA)**2

    R = tp_residual(chi)
    L += lam_tp * float(np.real(np.trace(R.conj().T @ R)))
    return L

def fit_chi_mle(dataset, lam_tp=1e4, maxiter=6000):
    theta0 = np.zeros(2*16, dtype=float)
    theta0[0] = 1.0  # seed

    res = minimize(
        lambda th: loss_for_theta(th, dataset, lam_tp=lam_tp),
        theta0,
        method="Nelder-Mead",
        options={"maxiter": maxiter, "xatol": 1e-10, "fatol": 1e-10}
    )
    chi = unpack_params_to_chi(res.x)
    alphas = best_alphas_for_chi(chi, dataset)
    return chi, alphas, res


#### Reporting + extracting a nearest unitary from $\chi$

A unitary process has a rank-1 $\chi$. We use:
- eigenvalues of $\chi$ as a “unitarity” indicator,
- the dominant eigenvector to build a unitary proxy $U_\text{fit}=\sum_k a_k\sigma_k$, and then project to nearest unitary via polar decomposition,
- axis–angle parameters $(\phi,\hat n)$ from $U_\text{fit}$.


In [14]:
def print_chi_fit(label, chi, alphas, res):
    chiH = make_hermitian(chi)
    print_header(label)
    print("optimizer success :", res.success)
    print("final loss        :", res.fun)
    print("Tr(chi)           :", float(np.real(np.trace(chiH))))
    print("eig(chi)          :", np.real_if_close(npl.eigvalsh(chiH)))
    print("\nalphas:")
    print_kv({k: float(v) for k, v in alphas.items()}, nd=6)
    print("\nchi (Re):\n", np.real(chiH))
    print("\nchi (Im):\n", np.imag(chiH))

def dominant_unitary_from_chi(chi):
    chiH = make_hermitian(chi)
    w, V = npl.eigh(chiH)
    idx = np.argmax(w)
    lam = float(np.real(w[idx]))
    a = V[:, idx]  
    
    U = np.zeros((2,2), dtype=complex)
    for ak, Sk in zip(a, paulis):
        U += ak * Sk

    M = U.conj().T @ U
    ew, eV = npl.eigh(make_hermitian(M))
    ew = np.maximum(ew, 1e-15)
    Minv_sqrt = (eV * (1/np.sqrt(ew))) @ eV.conj().T
    Uu = U @ Minv_sqrt
    return lam, a, Uu

def axis_angle_from_unitary(U):
    detU = npl.det(U)
    U_su2 = U / np.sqrt(detU)

    tr = np.trace(U_su2)
    c = np.clip(np.real(tr)/2.0, -1.0, 1.0)
    phi = 2*np.arccos(c)

    s = np.sin(phi/2)
    if abs(s) < 1e-12:
        return phi, np.array([np.nan, np.nan, np.nan])

    A = (U_su2 - U_su2.conj().T)/(2j)
    nx = np.real(np.trace(A @ X)) / (2*s)
    ny = np.real(np.trace(A @ Y)) / (2*s)
    nz = np.real(np.trace(A @ Z)) / (2*s)
    n = np.array([nx, ny, nz], dtype=float)
    n = n / (norm(n) + 1e-15)
    return phi, n


### (a) Show the results from performing quantum process tomography on the identity channel. Compare your resulting process matrix to the expected one using the process fidelity.

The assignment calls for reconstructing the identity process by performing process tomography with no optical element inserted.

However, in our dataset we did not record a complete HVDR 4×4 process-tomography table for the identity channel. Therefore a numerical reconstruction of $\chi_\text{id}$ and a process fidelity to the ideal identity cannot be reported here.

What we can still conclude:
- The QPT process is validated by the fact that it successfully reconstructs physically sensible channels for the GOOD and BAD HWP datasets below.
- In a complete dataset, the ideal identity process in the Pauli $\chi$ representation would be rank-1 with all weight on the identity component. Any deviation from that would quantify residual birefringence or misalignment even with no element inserted.


### (b) Repeat the analysis for the half wave plate rotated to 22.5 degrees.

We perform process tomography with the GOOD HWP inserted at $22.5^\circ$ and reconstruct $\chi$.

For an ideal HWP, the dominant coherent action corresponds to a Bloch-sphere rotation angle near $\phi\approx 180^\circ$. The fitted axis–angle parameters quantify how close the physical plate is to an ideal half-wave operation and whether the rotation axis matches the expected Hadamard-like orientation.


In [15]:
powers_good = {
    "H": {"H": 1.162, "V": 1.234, "D": 2.370, "R": 1.270},
    "V": {"H": 1.222, "V": 1.167, "D": 0.0035, "R": 1.202},
    "D": {"H": 2.373, "V": 0.0008, "D": 1.274, "R": 1.172},
    "R": {"H": 1.264, "V": 1.228, "D": 1.221, "R": 0.0029},
}

chi_good, alphas_good, res_good = fit_chi_mle(powers_good, lam_tp=1e4, maxiter=6000)
print_chi_fit("Q4(b) GOOD HWP @ 22.5° — reconstructed chi", chi_good, alphas_good, res_good)

lam_g, a_g, Uu_g = dominant_unitary_from_chi(chi_good)
phi_g, n_g = axis_angle_from_unitary(Uu_g)

print_header("Q4(b) GOOD HWP — unitary proxy parameters")
print("lambda_max =", lam_g)
print("phi (deg)  =", float(phi_g*180/np.pi))
print("axis n     =", n_g)



Q4(b) GOOD HWP @ 22.5° — reconstructed chi
optimizer success : False
final loss        : 454981.45457770344
Tr(chi)           : 1.0
eig(chi)          : [0.005799 0.033202 0.102938 0.858061]

alphas:
  H : 2.735739
  V : 2.261975
  D : 2.621302
  R : 2.855990

chi (Re):
 [[ 0.058877 -0.032951 -0.039639 -0.009199]
 [-0.032951  0.46896   0.030514  0.404534]
 [-0.039639  0.030514  0.044067 -0.00256 ]
 [-0.009199  0.404534 -0.00256   0.428095]]

chi (Im):
 [[ 0.       -0.013898  0.016917 -0.016085]
 [ 0.013898  0.       -0.025111  0.02583 ]
 [-0.016917  0.025111  0.        0.018392]
 [ 0.016085 -0.02583  -0.018392  0.      ]]

Q4(b) GOOD HWP — unitary proxy parameters
lambda_max = 0.8580606820358424
phi (deg)  = 183.27773380624384
axis n     = [-0.725611 -0.026232 -0.687605]


### Analysis 
The reconstructed process is largely unitary and close to an ideal half-wave action, but with a noticeable incoherent component.

#### How unitary-like is the channel?
The eigenvalues of the fitted $\chi$ are
$$(0.0058,\;0.0332,\;0.1029,\;0.8581).$$
A perfectly unitary channel corresponds to a rank-1 $\chi$. Here the dominant eigenvalue is
$$\lambda_{\max}\approx 0.858,$$
meaning about $86\%$ of the process weight is concentrated in a single coherent mode, while the remaining $\sim 14\%$ is spread across other modes. Operationally, this indicates the GOOD HWP implements a strong coherent transformation, but the experiment also introduces non-unitary “mixing” contributions.

#### What coherent rotation does it implement?
From the dominant eigenvector of $\chi$ we extract a nearest unitary $U_{\text{fit}}$ with axis–angle parameters
$$\phi \approx 183.28^\circ,\qquad \hat n \approx (-0.726,\,-0.026,\,-0.688).$$
For an ideal HWP, the expected Bloch rotation is $\phi\approx 180^\circ$. The reconstructed angle differs by only about $+3.3^\circ$, so the fitted coherent part is very close to the expected half-wave rotation.

The axis is predominantly in the $X$–$Z$ plane (the $Y$ component is small, $\approx -0.026$). This is physically consistent with a waveplate-type operation, since an ideal HWP corresponds to a $\pi$ rotation about an axis in the equatorial plane determined by the plate angle. At $22.5^\circ$, the ideal action is Hadamard-like. The fitted axis is close to such a bisector direction (roughly comparable $|X|$ and $|Z|$ components), indicating the coherent part of the GOOD plate is implementing the intended mapping.

#### Model fit quality
We have:
$$\alpha_H\approx 2.736,\;\alpha_V\approx 2.262,\;\alpha_D\approx 2.621,\;\alpha_R\approx 2.856.$$
This spread is expected when different prepared states couple through the apparatus with slightly different throughput. The key point is that these $\alpha$ values allow us to separate “how much light got through” from “what polarization transformation occurred.”

The optimizer did not report formal convergence (`success: False`), but the reconstructed $\chi$ is physical ($\mathrm{Tr}(\chi)=1$ and all eigenvalues nonnegative) and yields a consistent dominant coherent component.


### (c) Repeat the analysis for the half wave plate marked as “bad” rotated to an angle of your choice. Compare the ϕ value of the reconstruction to that of an ideal half wave plate.

We repeat the same reconstruction for the BAD HWP (dataset labelled $\phi=0^\circ$ in our notes).

We quantify "badness" by:
- comparing $\lambda_\max(\chi)$ to the GOOD HWP case (less rank-1 means less unitary-like),
- comparing the fitted rotation angle $\phi_\text{bad}$ to the ideal half-wave value $180^\circ$ via $\Delta\phi=\phi_\text{bad}-180^\circ$,
- and checking whether the rotation axis $\hat n$ is strongly distorted relative to the expected axis.


In [16]:
powers_bad = {
    "H": {"H": 1.718,  "V": 0.667,  "D": 2.177,  "R": 1.485},
    "V": {"H": 0.6575, "V": 1.746,  "D": 0.1728, "R": 0.968},
    "D": {"H": 2.224,  "V": 0.3766, "D": 0.770,  "R": 0.803},
    "R": {"H": 1.217,  "V": 1.257,  "D": 1.599,  "R": 0.04393},
}

chi_bad, alphas_bad, res_bad = fit_chi_mle(powers_bad, lam_tp=1e4, maxiter=6000)
print_chi_fit("Q4(c) BAD HWP — reconstructed chi", chi_bad, alphas_bad, res_bad)

lam_b, a_b, Uu_b = dominant_unitary_from_chi(chi_bad)
phi_b, n_b = axis_angle_from_unitary(Uu_b)

print_header("Q4(c) BAD HWP — unitary proxy parameters")
print("lambda_max =", lam_b)
print("phi (deg)  =", float(phi_b*180/np.pi))
print("axis n     =", n_b)

print_header("Q4(c) Deviation from ideal HWP rotation")
print("Delta phi (deg) = phi_bad - 180 =", float(phi_b*180/np.pi - 180.0))



Q4(c) BAD HWP — reconstructed chi
optimizer success : False
final loss        : 985452.2944551809
Tr(chi)           : 1.0
eig(chi)          : [0.002386 0.045753 0.138492 0.813369]

alphas:
  H : 2.312346
  V : 2.738441
  D : 3.101862
  R : 2.501564

chi (Re):
 [[ 0.050527 -0.031967 -0.000396  0.036185]
 [-0.031967  0.25416  -0.015869  0.284886]
 [-0.000396 -0.015869  0.043304  0.027126]
 [ 0.036185  0.284886  0.027126  0.652009]]

chi (Im):
 [[ 0.       -0.046121  0.009416 -0.052084]
 [ 0.046121  0.        0.036491 -0.020346]
 [-0.009416 -0.036491  0.       -0.056571]
 [ 0.052084  0.020346  0.056571  0.      ]]

Q4(c) BAD HWP — unitary proxy parameters
lambda_max = 0.8133692844468358
phi (deg)  = 170.04870329555095
axis n     = [0.462738 0.02043  0.886259]

Q4(c) Deviation from ideal HWP rotation
Delta phi (deg) = phi_bad - 180 = -9.95129670444905


Analysis

Compared to the GOOD HWP, the BAD HWP is both **less unitary-like** and **further from the ideal half-wave rotation angle**.

#### Reduced “unitarity” from the $\chi$ spectrum
The fitted $\chi$ eigenvalues are
$$(0.0024,\;0.0458,\;0.1385,\;0.8134).$$
The dominant eigenvalue is
$$\lambda_{\max}\approx 0.813,$$
smaller than the GOOD plate’s $\lambda_{\max}\approx 0.858$. This indicates the BAD plate has a larger incoherent/mixed contribution i.e. about $19\%$ outside the dominant coherent mode, consistent with a physically degraded waveplate. In other words, it is not well-described as “one clean unitary” to the same degree as the GOOD plate.

#### Coherent rotation angle and deviation from an ideal HWP
From the dominant mode we extract
$$\phi \approx 170.05^\circ,\qquad \hat n \approx (0.463,\;0.020,\;0.886).$$
An ideal half-wave plate corresponds to $\phi\approx 180^\circ$, so the deviation is
$$\Delta\phi = \phi - 180^\circ \approx -9.95^\circ.$$
This is a direct quantitative statement of retardance error: the coherent part of the BAD plate is closer to a $\sim 170^\circ$ rotation rather than a perfect $\pi$ rotation.

The fitted axis has a very small $Y$ component and is dominated by $Z$ with a moderate $X$ component. This again, looks waveplate-like with an axis in the $X$–$Z$ plane, but it is not the same axis as the GOOD HWP’s reconstruction. So the BAD plate differs in both *how much* it rotates and about what axis it rotates, which together imply a significantly altered polarization transformation.

#### Scale factors and robustness of interpretation
Here we have:
$$\alpha_H\approx 2.312,\;\alpha_V\approx 2.738,\;\alpha_D\approx 3.102,\;\alpha_R\approx 2.502.$$
As with the GOOD plate, variation in $\alpha$ reflects throughput differences rather than the gate action itself.

Although the optimizer again reports `success: False`, the fitted $\chi$ remains physical (nonnegative spectrum, $\mathrm{Tr}(\chi)=1$) and the extracted dominant coherent component provides a stable, interpretable comparison: relative to the GOOD plate, the BAD HWP is less unitary-like and has a $\sim 10^\circ$ shortfall from the ideal half-wave rotation angle.


## Q5 - Process Tomography of Trace-Preserving Non-Unitary Channels

In Experiment 5, we treat the calcite beam displacer as a *quantum channel* acting on the polarization qubit.
Because the displacer spatially separates the $|H\rangle$ and $|V\rangle$ components into different paths,
it can destroy polarization coherence (phase information) when we do not resolve/recombine the path degree of freedom.

This is not a unitary polarization rotation: it is expected to behave like a dephasing channel in the $H/V$ basis:
- populations in $H/V$ are largely preserved,
- off-diagonal coherence terms in the $H/V$ basis are suppressed.

We reconstruct the process matrix $\chi$ in the Pauli operator basis $\{I,X,Y,Z\}$ using maximum-likelihood
process tomography with per-input scale factors (because we measure powers, not probabilities).

Because we measure powers, not probabilities, we fit powers using per-input scale factors $\alpha_p$:
$$
P_{m|p} \approx \alpha_p\, p_{m|p},\qquad 
p_{m|p} = \mathrm{Tr}\!\left(\Pi_m\,\mathcal{E}_\chi(\rho_p)\right).
$$

We enforce a physical completely-positive map by parametrizing $\chi\succeq 0$ and use the trace-one convention $\mathrm{Tr}(\chi)=1$, so $\chi$ can be treated like a normalized Choi state.

For non-unitary processes, we compute the process fidelity by analogy with mixed-state fidelity, i.e., the Uhlmann/state-fidelity between normalized Choi matrices as described in the Notes.



### Experiment 5 HVDR power dataset

The following HVDR table (inputs $H,V,D,R$; analyzer settings $H,V,D,R$) is taken directly from the post-lab PDF.


In [17]:
powers_disp = {
    "H": {"H": 2.336,  "V": 0.00433, "D": 1.184, "R": 1.166},
    "V": {"H": 0.00765,"V": 2.357,   "D": 1.211, "R": 1.174},
    "D": {"H": 1.199,  "V": 1.149,   "D": 1.172, "R": 1.169},
    "R": {"H": 1.088,  "V": 1.262,   "D": 1.175, "R": 1.168},
}

preps = ["H","V","D","R"]
meas  = ["H","V","D","R"]


### (a) Present your reconstructed χ matrix for the calcite beam displacer and calculate the process fidelity. Note: use the process fidelity formula for non-unitary processes.

We reuse the same maximum-likelihood process tomography machinery as in Q4:
- $\chi\succeq 0$ and $\mathrm{Tr}(\chi)=1$ by parameterization,
- trace preservation enforced (or already built into your fitter),
- per-prep scale factors $\alpha_p$ are fit analytically.

After reconstructing $\chi_\mathrm{disp}$ we compare to the ideal **dephasing** channel in the $H/V$ basis.

In the Pauli basis, the ideal *complete dephasing* channel can be written as:
$$
\mathcal{E}_\mathrm{deph}(\rho) = \frac{1}{2}\left(\rho + Z\rho Z\right),
$$
which corresponds (in the trace-one $\chi$ convention) to a diagonal $\chi$ with weight on $I$ and $Z$ only:
$$
\chi_\mathrm{deph} = \mathrm{diag}\!\left(\frac12,\,0,\,0,\,\frac12\right).
$$

For non-unitary channels, process fidelity is computed as the mixed-state fidelity between the normalized Choi matrices:
$$
F_\mathrm{proc}(\chi,\chi_\mathrm{ideal}) =
\left(\mathrm{Tr}\sqrt{\sqrt{\chi_\mathrm{ideal}}\,\chi\,\sqrt{\chi_\mathrm{ideal}}}\right)^2.
$$

In [18]:
chi_disp, alphas_disp, res_disp = fit_chi_mle(powers_disp, lam_tp=1e4, maxiter=6000)
chi_disp = make_hermitian(chi_disp)
eigs_disp = np.real_if_close(npl.eigvalsh(chi_disp))

print_header("Q5(a) Beam displacer — reconstructed chi")
print("optimizer success :", res_disp.success)
print("final loss        :", res_disp.fun)
print("Tr(chi)           :", float(np.real(np.trace(chi_disp))))
print("eig(chi)          :", eigs_disp)

print("\nalphas:")
print_kv({k: float(v) for k, v in alphas_disp.items()}, nd=6)

print("\nchi (Re):\n", np.real(chi_disp))
print("\nchi (Im):\n", np.imag(chi_disp))

chi_deph = np.zeros((4,4), dtype=complex)
chi_deph[0,0] = 0.5  # I
chi_deph[3,3] = 0.5  # Z
chi_deph = make_hermitian(chi_deph)

F_proc = fidelity_uhlmann(chi_disp, chi_deph)

print_header("Q5(a) Non-unitary process fidelity vs ideal dephasing")
print("F_proc =", F_proc)



Q5(a) Beam displacer — reconstructed chi
optimizer success : False
final loss        : 831481.5597955383
Tr(chi)           : 1.0
eig(chi)          : [0.003875 0.044496 0.380358 0.571271]

alphas:
  H : 2.957223
  V : 2.228162
  D : 2.040064
  R : 3.497927

chi (Re):
 [[ 0.496984  0.033108 -0.101225 -0.069178]
 [ 0.033108  0.029552 -0.016007 -0.002636]
 [-0.101225 -0.016007  0.06518   0.039664]
 [-0.069178 -0.002636  0.039664  0.408285]]

chi (Im):
 [[ 0.        0.015149 -0.017872 -0.029013]
 [-0.015149  0.        0.000632  0.069395]
 [ 0.017872 -0.000632  0.        0.054148]
 [ 0.029013 -0.069395 -0.054148  0.      ]]

Q5(a) Non-unitary process fidelity vs ideal dephasing
F_proc = 0.8968003788612148


### Q5(a) Analysis (beam displacer channel)

The reconstructed $\chi$ confirms the beam displacer implements a strongly non-unitary, dephasing-like channel on polarization when the path degree of freedom is ignored.

#### Non-unitary signature from the $\chi$ eigenvalues
The eigenvalues of the reconstructed $\chi$ are:
$$(0.0039,\;0.0445,\;0.3804,\;0.5713).$$

A purely unitary channel would give a rank-1 $\chi$ (one eigenvalue near 1, others near 0). Here, instead, two eigenvalues are large ($\sim 0.57$ and $\sim 0.38$), showing the channel has a substantial incoherent/mixed component and cannot be described by a single unitary operation. This is exactly what we expect if polarization becomes correlated with an unobserved path mode.

#### Structure of $\chi$
For ideal complete dephasing in the $H/V$ basis, the trace-one Pauli-basis process matrix is:
$$\chi_\mathrm{deph} = \mathrm{diag}\!\left(\frac12,\,0,\,0,\,\frac12\right),$$
meaning only the $I$ and $Z$ components appear.

Our reconstructed diagonal entries are:
- $\chi_{II}\approx 0.497$ (very close to $0.5$),
- $\chi_{ZZ}\approx 0.408$ (substantial, but below $0.5$),
while the $X$ and $Y$ diagonal weights remain much smaller:
- $\chi_{XX}\approx 0.0296$,
- $\chi_{YY}\approx 0.0652$.

This matches the physical picture: the dominant action is suppression of coherence between $|H\rangle$ and $|V\rangle$, which is precisely a $Z$-basis dephasing effect. The fact that $\chi_{ZZ}$ is not exactly $0.5$ and that small $X/Y$ weights appear indicates additional imperfections beyond ideal dephasing (e.g., slight polarization rotation, imbalance, or residual coupling effects).

#### Process fidelity to ideal dephasing
The non-unitary process fidelity to the ideal complete-dephasing channel is:
$$F_\mathrm{proc}(\chi_\mathrm{disp},\chi_\mathrm{deph}) \approx 0.8968.$$

A value close to 1 would indicate nearly perfect dephasing. Here, $\sim 0.90$ means the channel is mostly dephasing-like, but with measurable deviations. This is consistent with a realistic beam-displacer implementation where the dominant effect is loss of $H/V$ coherence, plus smaller additional non-ideal effects.

#### Per-preparation scale factors
We have:
$$\alpha_H\approx 2.957,\;\alpha_V\approx 2.228,\;\alpha_D\approx 2.040,\;\alpha_R\approx 3.498.$$
These reflect throughput differences between preparations (alignment, coupling, power drift) and are not themselves the gate action. Including $\alpha_p$ is important because it lets the fit focus on the *relative* measurement statistics that encode the channel.

#### Note on `optimizer success : False`
Although the optimizer did not report formal convergence, the reconstructed $\chi$ is still normalized ($\mathrm{Tr}(\chi)=1$) and has a nonnegative spectrum, and the qualitative conclusions are robust: the channel is clearly non-unitary and is close to a dephasing map, with fidelity $\approx 0.90$ to the ideal dephasing channel.


### (b) What is the name for the quantum channel the beam displacer performs? What is the Kraus representation of this channel?

### Analysis

#### Channel identification from $\chi$.
Our reconstructed process matrix has a dominant diagonal weight on the $I$ component ($\chi_{II}\approx 0.497$) and a large weight on the $Z$ component ($\chi_{ZZ}\approx 0.408$), while the $X$ and $Y$ components are much smaller. This is exactly the pattern expected for dephasing in the $H/V$ basis, because $Z$ is the operator that flips the relative phase between $|H\rangle$ and $|V\rangle$ without changing their populations.

#### Operational meaning: what the channel does to a general polarization state.
Write a general qubit state in the $H/V$ basis as
$$
\rho=\begin{pmatrix}
\rho_{HH} & \rho_{HV}\\
\rho_{VH} & \rho_{VV}
\end{pmatrix}.
$$
A dephasing channel preserves the populations ($\rho_{HH},\rho_{VV}$) but suppresses the coherence ($\rho_{HV},\rho_{VH}$). In the complete-dephasing limit, the off-diagonal terms are driven to zero in one application:
$$
\rho \longmapsto \begin{pmatrix}
\rho_{HH} & 0\\
0 & \rho_{VV}
\end{pmatrix}.
$$
This matches the physical mechanism of the beam displacer: it correlates $|H\rangle$ and $|V\rangle$ with orthogonal path modes. If we do not resolve or erase the path information, the relative phase between $|H\rangle$ and $|V\rangle$ becomes inaccessible, which appears as loss of off-diagonal coherence in polarization.

#### Kraus representation and why it matches the $I/Z$ structure.
A standard Kraus form for dephasing is
$$
K_0=\sqrt{1-p}\,I,\qquad K_1=\sqrt{p}\,Z,
$$
so that
$$
\mathcal{E}(\rho)=(1-p)\rho + p\,Z\rho Z.
$$
This form makes the connection to the Pauli-basis $\chi$ intuitive: the channel is a probabilistic mixture of “do nothing” ($I$) and “apply a $Z$ phase flip” ($Z$). In the trace-one $\chi$ convention, the ideal complete dephasing channel corresponds to equal weights on $I$ and $Z$:
$$
\chi_\mathrm{deph}=\mathrm{diag}\!\left(\frac12,0,0,\frac12\right),
$$
which is precisely the target used in Q5(a) to compute $F_\mathrm{proc}$.

#### Why our channel is close but not perfect dephasing.
The measured fidelity $F_\mathrm{proc}\approx 0.8968$ shows the beam displacer is strongly dephasing-like but not identical to the ideal channel. This is consistent with the reconstructed $\chi$ having small additional $X/Y$ weights and off-diagonal terms, which represent extra imperfections beyond pure coherence loss superimposed on the dominant dephasing behavior.


## Q6 — Process Tomography of a Non-Trace Preserving Channel

In Experiment 6 we characterize a partial polarizer, which is a non-trace preserving operation because it has polarization-dependent transmission i.e. state-dependent loss.

### Experiment 6 HVDR dataset (from the post-lab PDF)

Inputs: $H,V,D,R$  
Measurements: $H,V,D,R$  
All values are measured optical powers (mW), proportional to probabilities up to a per-prep scale factor.


In [19]:
powers_exp6 = {
    "H": {"H": 1.355,   "V": 0.684, "D": 0.642, "R": 0.620},
    "V": {"H": 0.00354, "V": 1.925, "D": 1.016, "R": 0.933},
    "D": {"H": 0.705,   "V": 0.960, "D": 1.583, "R": 0.756},
    "R": {"H": 0.644,   "V": 0.924, "D": 0.934, "R": 1.600},
}


### (a) Show your results from performing quantum process tomography on the partial polarizer.What strength of the partial polarizer did you measure? Explain how you obtained this value from your |H⟩ and |V ⟩ transmission measurements.


We define the strength $a$ as the relative transmission of the two eigenpolarizations (in the lab convention where $V$ is the higher-transmission axis):

$$
a_\mathrm{meas}=\frac{T_H}{T_V}\approx \frac{P(H\to H)}{P(V\to V)}.
$$

This ratio is “clean” because it compares like-with-like: both are measured in a prep = analyze configuration, so overall power scale factors cancel.


In [20]:
P_HH = powers_exp6["H"]["H"]
P_VV = powers_exp6["V"]["V"]

a_meas = P_HH / P_VV

print_header("Q6(a) Partial polarizer strength from direct transmissions")
print("P(H->H) =", P_HH, "mW")
print("P(V->V) =", P_VV, "mW")
print("a_meas = P(H->H)/P(V->V) =", a_meas)



Q6(a) Partial polarizer strength from direct transmissions
P(H->H) = 1.355 mW
P(V->V) = 1.925 mW
a_meas = P(H->H)/P(V->V) = 0.7038961038961039


### Analysis

Using the direct transmission method,
$$a_\text{meas}=\frac{T_H}{T_V}\approx\frac{P(H\to H)}{P(V\to V)},$$
we obtain:
- $P(H\to H)=1.355$ mW
- $P(V\to V)=1.925$ mW
so
$$a_\text{meas}= \frac{1.355}{1.925}\approx 0.7039.$$

This means the device transmits $H$ at about $70\%$ of the $V$ transmission (in our lab convention where $V$ is the higher-transmission eigenpolarization). Since $a_\text{meas}<1$, the element is clearly a **partial polarizer** rather than a unitary waveplate: it introduces state-dependent loss.


### (b) Show the expected form of the process matrix for an arbitrary strength partial polarizer, characterized by strength, a. Find the value of a that gives the theoretical matrix closest to your experimental result and compare this to the polarizer strength you measured directly. Use the process fidelity for non-trace preserving processes to characterize the success of the reconstruction.

Model the partial polarizer as a single-Kraus filter in the $H/V$ basis:
$$
K(a) = |V⟩⟨V| + \sqrt{a}\,|H⟩⟨H|,\qquad 0\le a\le 1.
$$

Given $K$, expand it in the Pauli basis:
$$
K = \sum_{k\in\{I,X,Y,Z\}} c_k\,\sigma_k,\qquad c_k=\frac12\mathrm{Tr}(\sigma_k K).
$$

For a single Kraus operator, the process matrix is rank-1:
$$
\chi_{ij} = c_i\,c_j^*.
$$

We compute $\chi_\mathrm{th}(a)$, then choose $a$ that best matches the experimentally reconstructed $\chi_\mathrm{exp}$.

Because the process is non-trace preserving, we use a trace-normalized process fidelity:
$$
F_\mathrm{NTP}(\chi_1,\chi_2)=
\frac{\left(\mathrm{Tr}\sqrt{\sqrt{\chi_1}\,\chi_2\,\sqrt{\chi_1}}\right)^2}{\mathrm{Tr}(\chi_1)\,\mathrm{Tr}(\chi_2)}.
$$

(With the trace-one convention used in the reconstruction, the denominator is 1, but we keep the formula explicit.)


In [21]:
chi_exp6, alphas_exp6, res_exp6 = fit_chi_mle(powers_exp6, lam_tp=0.0, maxiter=6000)
chi_exp6 = make_hermitian(chi_exp6)

eigs_exp6 = np.real_if_close(npl.eigvalsh(chi_exp6))

print_header("Q6(b) Experimental reconstruction (non-TP): chi_exp6")
print("optimizer success :", res_exp6.success)
print("final loss        :", res_exp6.fun)
print("Tr(chi_exp6)      :", float(np.real(np.trace(chi_exp6))))
print("eig(chi_exp6)     :", eigs_exp6)

print("\nalphas:")
print_kv({k: float(v) for k, v in alphas_exp6.items()}, nd=6)

print("\nchi_exp6 (Re):\n", np.real(chi_exp6))
print("\nchi_exp6 (Im):\n", np.imag(chi_exp6))



Q6(b) Experimental reconstruction (non-TP): chi_exp6
optimizer success : False
final loss        : 869721.062125607
Tr(chi_exp6)      : 1.0
eig(chi_exp6)     : [0.009584 0.034007 0.141228 0.815181]

alphas:
  H : 1.745620
  V : 1.964175
  D : 1.664433
  R : 2.057075

chi_exp6 (Re):
 [[ 0.802944  0.021876 -0.066645 -0.059641]
 [ 0.021876  0.080996 -0.052185 -0.018788]
 [-0.066645 -0.052185  0.075756  0.009866]
 [-0.059641 -0.018788  0.009866  0.040304]]

chi_exp6 (Im):
 [[ 0.        0.013464 -0.000743 -0.001764]
 [-0.013464  0.        0.040602  0.000053]
 [ 0.000743 -0.040602  0.        0.007133]
 [ 0.001764 -0.000053 -0.007133  0.      ]]


In [22]:
def chi_from_single_kraus(K):
    coeffs = np.array([0.5*np.trace(S @ K) for S in paulis], dtype=complex)
    chi = np.outer(coeffs, np.conjugate(coeffs))
    # normalize to Tr(chi)=1 to match our reconstruction convention
    tr = float(np.real(np.trace(chi)))
    return chi / (tr if tr > 1e-15 else 1.0)

def chi_th_partial(a):
    K = projs_6["V"] + np.sqrt(a) * projs_6["H"]
    return make_hermitian(chi_from_single_kraus(K))

def frob_dist(A, B):
    return float(np.linalg.norm(A - B))

def fidelity_ntp(chi1, chi2):
    chi1h = make_hermitian(chi1)
    chi2h = make_hermitian(chi2)
    num = fidelity_uhlmann(chi1h, chi2h)
    den = float(np.real(np.trace(chi1h)) * np.real(np.trace(chi2h)))
    return float(num / den) if den > 1e-15 else 0.0

grid = np.linspace(0.0, 1.0, 2001)
dists = np.array([frob_dist(chi_exp6, chi_th_partial(a)) for a in grid])
a_fit = float(grid[int(np.argmin(dists))])

chi_th = chi_th_partial(a_fit)
F_ntp = fidelity_ntp(chi_exp6, chi_th)

print_header("Q6(b) Best-fit theoretical partial polarizer")
print("a_fit =", a_fit)
print("||chi_exp6 - chi_th(a_fit)||_F =", float(np.min(dists)))
print("F_NTP =", F_ntp)

print("\nchi_th(a_fit) (Re):\n", np.real(chi_th))
print("\nchi_th(a_fit) (Im):\n", np.imag(chi_th))



Q6(b) Best-fit theoretical partial polarizer
a_fit = 0.7325
||chi_exp6 - chi_th(a_fit)||_F = 0.26561795278211303
F_NTP = 0.8075802711774205

chi_th(a_fit) (Re):
 [[ 0.994004  0.        0.       -0.077201]
 [ 0.        0.        0.        0.      ]
 [ 0.        0.        0.        0.      ]
 [-0.077201  0.        0.        0.005996]]

chi_th(a_fit) (Im):
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


### Analysis
#### Experimental process matrix $\chi_\text{exp6}$: evidence of non-unitary/non-ideal filtering
The reconstructed $\chi_\text{exp6}$ has eigenvalues
$$(0.0096,\;0.0340,\;0.1412,\;0.8152).$$

A perfect single-Kraus filter would produce an rank-1 $\chi$ (one eigenvalue near 1). Here the dominant eigenvalue is
$$\lambda_{\max}\approx 0.815,$$
with significant additional weight in subdominant modes, notably a second eigenvalue $\approx 0.141$. This means the channel is strongly dominated by one filter-like action, but the experimental process contains additional non-ideal contributions beyond the simplest one-parameter model.

The fitted per-preparation scale factors
$$\alpha_H\approx 1.746,\;\alpha_V\approx 1.964,\;\alpha_D\approx 1.664,\;\alpha_R\approx 2.057$$
indicate that overall output differs between prepared inputs, which is expected experimentally and is separated from the channel action by explicitly fitting $\alpha_p$.

#### Theoretical model and best-fit strength
We model the partial polarizer as a diagonal filter in the $H/V$ basis with a single Kraus operator
$$
K(a)=|V⟩⟨V|+\sqrt{a}\,|H⟩⟨H|,\qquad 0\le a\le 1.
$$
This model predicts a $\chi_\text{th}(a)$ that has support only in the $I/Z$ sector of the Pauli basis because a diagonal $H/V$ operator contains only $I$ and $Z$ components, not $X$ or $Y$.

Fitting this model to the experimentally reconstructed $\chi_\text{exp6}$ gives:
$$a_\text{fit}=0.7325.$$

This is close to the direct transmission estimate $a_\text{meas}=0.7039$, differing by
$$a_\text{fit}-a_\text{meas}\approx 0.0286,$$
which is about a $4.1\%$ relative increase compared to $a_\text{meas}$. The agreement in scale indicates that the dominant behavior is indeed polarization-dependent attenuation consistent with a partial polarizer, while the discrepancy suggests additional non-ideal effects that are not captured by a single scalar $a$.

The fitted theoretical matrix has the expected “filter-like” structure (only $I/Z$ block nonzero), e.g. a large $\chi_{II}$ term and small $I\!-\!Z$ correlations, with zeros in the $X$ and $Y$ rows/columns.

#### Non-trace-preserving process fidelity
To quantify agreement we compute the non-trace-preserving process fidelity:
$$
F_\text{NTP}(\chi_\text{exp6},\chi_\text{th}(a_\text{fit}))=
\frac{\left(\mathrm{Tr}\sqrt{\sqrt{\chi_\text{th}}\,\chi_\text{exp6}\,\sqrt{\chi_\text{th}}}\right)^2}
{\mathrm{Tr}(\chi_\text{exp6})\,\mathrm{Tr}(\chi_\text{th})}.
$$

With our trace-one convention, the denominator is 1, and the computed value is:
$$F_\text{NTP}\approx 0.8076.$$

A fidelity around $0.81$ means the ideal single-parameter partial-polarizer model captures most of the observed process, but not all of it. This is consistent with the eigenvalue spectrum of $\chi_\text{exp6}$ (not rank-1) and indicates that the experimental device behaves like a partial polarizer plus additional imperfections (small coherent distortions and/or extra mixture) beyond pure diagonal filtering.
